# MMLU -- Inference, Evaluation & Contrastive Training

This notebook runs the full HalluLens pipeline on MMLU in **closed-book mode**
(no answer choices provided -- the model relies on parametric memory).

**Dataset:** [MMLU](https://huggingface.co/datasets/cais/mmlu) -- Massive Multitask Language Understanding
- `test` split: 14,079 questions (default)
- `validation` split: 1,540 questions
- `auxiliary_train` split: ~99,800 questions
- Filtered to **factual-recall subjects** only (history, geography, biology, medicine, law, etc.)
- Reasoning-heavy subjects excluded (abstract_algebra, formal_logic, mathematics, etc.)

**Why MMLU for hallucination detection:**
MMLU covers dozens of academic domains. By filtering to factual subjects, we test whether
hallucination detection generalises across many knowledge areas simultaneously -- a broader
evaluation than single-domain benchmarks like TriviaQA or NQ.

**Steps:**
1. Inference -- generate model responses with activation logging
2. Evaluation -- substring match against gold answer; write `eval_results.json`
3. Inspection -- per-sample view, breakdown by subject

**Prerequisite:** A running vLLM server (`COMPUTE_CONTEXT=REMOTE_GPU` in `.env`).

## 1 -- Configuration

In [ ]:
import os, sys, json
from pathlib import Path

# -- Repo root on path
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

# -- Model
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
model_name = MODEL.split("/")[-1]

# -- Dataset split and sample count
SPLIT = "test"       # "test" | "validation" | "auxiliary_train"
N     = None          # None = entire filtered split

# -- Storage paths
base_dir          = Path("shared/mmlu")
activations_path  = str(base_dir / "activations.zarr")
log_file          = str(base_dir / "server.log")

output_dir        = "output"
task_dir          = Path(output_dir) / "mmlu" / model_name
generations_file  = str(task_dir / "generation.jsonl")
eval_results_file     = str(task_dir / "eval_results.json")
eval_results_llm_file = str(task_dir / "eval_results_llm.json")

# -- Activation logging
LOGGER_TYPE = "zarr"

# -- Inference settings
MAX_TOKENS  = 64
TEMPERATURE = 0.0

# -- Batched inference (ModelAdapter path)
BATCH_SIZE = 8

# -- LLM evaluator
LLM_EVALUATOR = None   # None -> use class default

base_dir.mkdir(parents=True, exist_ok=True)
task_dir.mkdir(parents=True, exist_ok=True)

print(f"Model             : {MODEL}")
print(f"Split             : {SPLIT}  (N={N or 'all'})")
print(f"Batch size        : {BATCH_SIZE or 'disabled (sequential server path)'}")
print(f"Generations file  : {generations_file}")
print(f"Eval (substring)  : {eval_results_file}")
print(f"Eval (LLM judge)  : {eval_results_llm_file}")
print(f"Activations       : {activations_path}")
print(f"Logger type       : {LOGGER_TYPE}")

## 2 -- Inference

Generates model responses for MMLU questions in closed-book mode and logs activations.
Resume-safe: questions already in `generation.jsonl` are skipped.

When `BATCH_SIZE` is set, inference uses `HFTransformersAdapter.infer_batch()` directly --
no HTTP server is needed.

In [ ]:
from scripts.run_with_server import run_experiment

run_experiment(
    step="inference",
    task="mmlu",
    model=MODEL,
    split=SPLIT,
    N=N,
    logger_type=LOGGER_TYPE,
    activations_path=activations_path,
    log_file=log_file,
    output_dir=output_dir,
    generations_file_path=generations_file,
    max_inference_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    resume=True,
    batch_size=BATCH_SIZE,
)

## 3 -- Evaluation

Two evaluation methods:

- **Substring match** (`eval`) -- fast, deterministic. Correct if the gold answer appears verbatim in the response.
- **LLM judge** (`eval_llm`) -- uses an LLM to assess semantic correctness. Handles paraphrased answers.

In [ ]:
from scripts.run_with_server import run_experiment

# -- 3a: Substring match eval
run_experiment(
    step="eval",
    task="mmlu",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_file,
)

# -- 3b: LLM-judge eval
run_experiment(
    step="eval_llm",
    task="mmlu",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_llm_file,
    llm_evaluator=LLM_EVALUATOR,
    resume=True,
)

## 4 -- Results Inspection & Per-Subject Analysis

In [ ]:
import pandas as pd

with open(eval_results_file) as f:
    substr_res = json.load(f)

n = substr_res["total_count"]
substr_correct_count = substr_res["accurate_count"]

print(f"Model          : {model_name}")
print(f"Split          : {SPLIT}  (N={n})")
print()
print(f"{'Metric':<30} {'Substring':>12}")
print("-" * 43)
print(f"{'Correct count':<30} {substr_correct_count:>12}")
print(f"{'Correct rate':<30} {substr_res['correct_rate']:>12.1%}")
print(f"{'Hallucination rate':<30} {substr_res['halu_Rate']:>12.1%}")

In [ ]:
# Build per-sample dataframe
raw_df = pd.read_json(task_dir / "raw_eval_res.jsonl", lines=True)

print(f"Loaded {len(raw_df)} samples")
raw_df.head()

In [ ]:
# -- Hallucination rate by subject
if "subject" in raw_df.columns:
    subject_stats = (
        raw_df.groupby("subject")
        .agg(
            count=("is_hallucination", "count"),
            halu_rate=("is_hallucination", "mean"),
            correct_rate=("is_correct", "mean"),
        )
        .reset_index()
        .sort_values("halu_rate", ascending=False)
    )
    for col in ["halu_rate", "correct_rate"]:
        subject_stats[col] = subject_stats[col].map("{:.1%}".format)
    print("Hallucination rate by subject (sorted by halu rate):")
    display(subject_stats)

In [ ]:
# -- Subject category grouping
CATEGORY_MAP = {
    "History": ["high_school_european_history", "high_school_us_history",
                "high_school_world_history", "prehistory"],
    "Geography": ["high_school_geography"],
    "Biology & Health": ["anatomy", "high_school_biology", "college_biology",
                         "clinical_knowledge", "medical_genetics", "nutrition",
                         "virology", "college_medicine", "professional_medicine"],
    "Physical Science": ["high_school_chemistry", "college_chemistry",
                          "high_school_physics", "college_physics", "astronomy"],
    "Law": ["international_law", "jurisprudence", "professional_law"],
    "Social Science": ["high_school_government_and_politics", "high_school_macroeconomics",
                        "high_school_microeconomics", "high_school_psychology",
                        "sociology", "us_foreign_policy", "public_relations",
                        "human_sexuality", "world_religions", "philosophy",
                        "moral_disputes", "moral_scenarios", "business_ethics"],
    "Other": ["computer_security", "management", "marketing", "miscellaneous"],
}

subj_to_cat = {}
for cat, subjects in CATEGORY_MAP.items():
    for s in subjects:
        subj_to_cat[s] = cat

if "subject" in raw_df.columns:
    raw_df["category"] = raw_df["subject"].map(subj_to_cat).fillna("Other")
    cat_stats = (
        raw_df.groupby("category")
        .agg(
            count=("is_hallucination", "count"),
            halu_rate=("is_hallucination", "mean"),
            correct_rate=("is_correct", "mean"),
        )
        .reset_index()
        .sort_values("halu_rate", ascending=False)
    )
    for col in ["halu_rate", "correct_rate"]:
        cat_stats[col] = cat_stats[col].map("{:.1%}".format)
    print("Hallucination rate by subject category:")
    display(cat_stats)

In [ ]:
# -- Example hallucinated responses
halu_df = raw_df[raw_df["is_hallucination"]].reset_index(drop=True)
print(f"Total hallucinations: {len(halu_df)} / {len(raw_df)}")
print()
for i, row in halu_df.head(8).iterrows():
    print(f"[{row.get('subject', '')}]")
    print(f"  Q  : {row['question'][:120]}")
    print(f"  Gen: {row['generation'][:120]}")
    print(f"  Ans: {row['answer']}")
    print()